# Hung #39: Development verification on Colab

This thin notebook calls a pinned repository checkout. Set the code bundle path/hash, code SHA and RUN_ID below, then run on a fresh Colab runtime. Re-running with the same RUN_ID reuses only complete, hash-verified stages. The public DATA-v1 input archive is downloaded and verified. The configuration is diagnostic and requires team review before acceptance. Final/test data is never read.


In [ ]:
import hashlib
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CODE_BUNDLE_ON_DRIVE = Path("")  # existing Git bundle containing this reviewed branch
EXPECTED_BUNDLE_SHA256 = ""
EXPECTED_CODE_SHA = ""
RUN_ID = ""  # unique, stable name; reuse only to continue the same run
DRIVE_RUN_PARENT = Path("/content/drive/MyDrive/hung-sprint2-runs")
DEVICE = "cuda"  # set cpu only as a separate declared run configuration
PHASE = "verification"  # representative requires team acceptance first
BUDGETS = (7, 14, 28)
SEEDS = (7, 11, 19)
assert RUN_ID and EXPECTED_CODE_SHA and len(EXPECTED_BUNDLE_SHA256) == 64
assert CODE_BUNDLE_ON_DRIVE.is_file()
assert PHASE in {"verification", "representative"} and DEVICE in {"cpu", "cuda"}


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if sha256(CODE_BUNDLE_ON_DRIVE) != EXPECTED_BUNDLE_SHA256:
    raise ValueError("Code bundle hash mismatch")
repo = Path("/content/hung-sprint2-code")
if repo.exists():
    raise FileExistsError(repo)
subprocess.run(["git", "clone", str(CODE_BUNDLE_ON_DRIVE), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", EXPECTED_CODE_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
if actual_sha != EXPECTED_CODE_SHA:
    raise ValueError("Code commit mismatch")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-colab.txt")], check=True
)
wheel_index = (
    "https://download.pytorch.org/whl/cu126"
    if DEVICE == "cuda"
    else "https://download.pytorch.org/whl/cpu"
)
requirements = repo / ("requirements-cuda.txt" if DEVICE == "cuda" else "requirements-model.txt")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--index-url", wheel_index, "-r", str(requirements)],
    check=True,
)
probe = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    text=True,
).strip()
if not probe.startswith("2.8.0+" + ("cu126" if DEVICE == "cuda" else "cpu")) or (
    DEVICE == "cuda" and not probe.endswith("True")
):
    raise RuntimeError("Pinned PyTorch device probe failed: " + probe)
print("code", actual_sha, "torch probe", probe, "device", DEVICE)

In [ ]:
import gdown

local_archive = Path("/content/inputs-urbanev-SZH-v1.zip")
if local_archive.exists():
    raise FileExistsError(local_archive)
gdown.download(id="17pfYqnnSq0Uao2zDNdb8h7GAYfotu_w4", output=str(local_archive), quiet=False)
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(local_archive),
        "--package",
        "inputs-urbanev-SZH-v1.zip",
    ],
    cwd=repo,
    check=True,
)
input_root = Path("/content/hung-39-input")
if input_root.exists():
    raise FileExistsError(input_root)
input_root.mkdir()
with zipfile.ZipFile(local_archive) as archive:
    for member in archive.infolist():
        target = (input_root / member.filename).resolve()
        if not target.is_relative_to(input_root.resolve()) or target.exists():
            raise ValueError("Unsafe or duplicate ZIP member")
    archive.extractall(input_root)
input_base = input_root / "data/inputs/issue18-input-handoff-v1-20261019"
output_root = DRIVE_RUN_PARENT / RUN_ID
command = [
    sys.executable,
    "-m",
    "experiments.hung_sprint2.run_development",
    "--input-base",
    str(input_base),
    "--selection",
    str(repo / "docs/reports/hung-development-selection-20261025-v2.json"),
    "--config",
    str(repo / "experiments/hung_sprint2/development_config_v1.json"),
    "--output-root",
    str(output_root),
    "--phase",
    PHASE,
    "--device",
    DEVICE,
    "--budgets",
    *map(str, BUDGETS),
    "--seeds",
    *map(str, SEEDS),
    "--execute",
]
subprocess.run(command, cwd=repo, check=True)
print("matrix identity:", output_root / "matrix_identity.json")
print("Run folder:", output_root)